# Phikon v1 + linear head + multi-seed ensemble
# Baseline vs FroFA (feature-space augmentation), same precomputed features

In [ ]:
!pip install -q transformers torchmetrics

In [ ]:
import h5py, torch, torch.nn as nn, torch.nn.functional as F
import random, os, glob, gc, time, hashlib
import numpy as np, pandas as pd, torchmetrics
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm
from transformers import ViTModel

SEED = 0
torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)

# config
BATCH_SIZE = 32
N_SEEDS = 5
TTA_N = 3
LR = 1e-3
WEIGHT_DECAY = 1e-3
NUM_EPOCHS = 100
PATIENCE = 10

# FroFA params
FEAT_MIXUP_ALPHA = 0.2
FEAT_DROPOUT = 0.1
FEAT_NOISE_STD = 0.01

# TTA transforms (prediction only)
_tta = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
])


class H5Dataset(Dataset):
    def __init__(self, path, mode='train'):
        self.mode = mode
        self.imgs, self.labels, self.centers, self.img_ids = [], [], [], []
        with h5py.File(path, 'r') as hdf:
            for k in list(hdf.keys()):
                self.imgs.append(np.array(hdf[k]['img']))
                self.img_ids.append(int(k))
                if mode == 'train':
                    self.labels.append(int(np.array(hdf[k]['label'])))
                    self.centers.append(int(np.array(hdf[k]['metadata'])[0]))
                else:
                    self.labels.append(-1); self.centers.append(0)
        self.imgs = np.stack(self.imgs)
        self.needs_rescale = float(self.imgs.max()) > 1.5
    def __len__(self): return len(self.imgs)
    def __getitem__(self, idx):
        img = torch.from_numpy(self.imgs[idx]).float()
        if self.needs_rescale: img = img / 255.0
        return img, self.labels[idx], self.centers[idx], self.img_ids[idx]


class PrecomputedDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels.unsqueeze(-1).float()
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx): return self.features[idx], self.labels[idx]


@torch.no_grad()
def precompute(dataloader, model, transform, device):
    all_f, all_l = [], []
    for imgs, labels, _, _ in tqdm(dataloader, desc='Extracting features'):
        batch = torch.stack([transform(img) for img in imgs]).to(device)
        out = model(pixel_values=batch)
        all_f.append(out.last_hidden_state[:, 0, :].cpu())
        all_l.append(labels)
    return torch.cat(all_f), torch.cat(all_l)


# feature-space augmentation (FroFA)

def feature_mixup(feats, labels, alpha=0.2):
    if alpha <= 0: return feats, labels
    lam = max(np.random.beta(alpha, alpha), 0.5)
    idx = torch.randperm(len(feats))
    return lam * feats + (1 - lam) * feats[idx], lam * labels + (1 - lam) * labels[idx]

def feature_augment(feats, labels, dropout_p=0.1, noise_std=0.01, mixup_alpha=0.2):
    feats, labels = feature_mixup(feats, labels, mixup_alpha)
    feats = F.dropout(feats, p=dropout_p, training=True)
    if noise_std > 0:
        feats = feats + torch.randn_like(feats) * noise_std
    return feats, labels


def train_head(tf, tl, vf, vl, feat_dim, device, seed=0,
               lr=1e-3, wd=1e-3, bs=32, epochs=100, patience=10,
               mixup_alpha=0.0, feat_dropout=0.0, feat_noise=0.0,
               save='/kaggle/working/best_model.pth'):
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    use_aug = (mixup_alpha > 0 or feat_dropout > 0 or feat_noise > 0)

    tloader = DataLoader(PrecomputedDataset(tf, tl), shuffle=True, batch_size=bs)
    vloader = DataLoader(PrecomputedDataset(vf, vl), shuffle=False, batch_size=bs)

    head = nn.Sequential(nn.Linear(feat_dim, 1), nn.Sigmoid()).to(device)
    opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=wd)
    crit = nn.BCELoss()
    met = torchmetrics.Accuracy('binary')

    best_loss, best_acc, best_ep = float('inf'), 0.0, 0
    for ep in range(epochs):
        head.train()
        tL, tA = [], []
        for f, l in tloader:
            f, l = f.to(device), l.to(device)
            if use_aug:
                f_aug, l_aug = feature_augment(f, l, feat_dropout, feat_noise, mixup_alpha)
            else:
                f_aug, l_aug = f, l
            opt.zero_grad()
            p = head(f_aug)
            loss = crit(p, l_aug)
            loss.backward()
            opt.step()
            with torch.no_grad():
                p_clean = head(f)
            tL.append(loss.item())
            tA.append(met(p_clean.cpu(), (l.cpu() > 0.5).int()).item())

        head.eval()
        vL, vA = [], []
        for f, l in vloader:
            f, l = f.to(device), l.to(device)
            with torch.no_grad():
                p = head(f)
            vL.append(crit(p, l).item())
            vA.append(met(p.cpu(), l.int().cpu()).item())

        vl_m, va_m = np.mean(vL), np.mean(vA)
        print(f'Ep [{ep+1}/{epochs}] T: loss={np.mean(tL):.4f} acc={np.mean(tA):.4f} | V: loss={vl_m:.4f} acc={va_m:.4f}')
        if vl_m < best_loss:
            best_loss, best_acc, best_ep = vl_m, va_m, ep
            torch.save(head.state_dict(), save)
        if ep - best_ep >= patience:
            break

    head.load_state_dict(torch.load(save, weights_only=True))
    return head, {'seed': seed, 'best_ep': best_ep + 1, 'val_loss': best_loss, 'val_acc': best_acc}


def print_summary(name, metrics):
    accs = [m['val_acc'] for m in metrics]
    losses = [m['val_loss'] for m in metrics]
    print(f'\n{name} -- {len(metrics)} heads')
    print(f'{"Seed":>6} {"BestEp":>8} {"ValLoss":>10} {"ValAcc":>10}')
    for m in metrics:
        print(f'{m["seed"]:>6} {m["best_ep"]:>8} {m["val_loss"]:>10.4f} {m["val_acc"]:>10.4f}')
    print(f'{"Mean":>6} {"":>8} {np.mean(losses):>10.4f} {np.mean(accs):>10.4f}')
    print(f'{"Std":>6} {"":>8} {np.std(losses):>10.4f} {np.std(accs):>10.4f}')


def predict_ensemble(model, transform, heads, test_path, device,
                     tta_n=3, bs=64, save='/kaggle/working/submission.csv'):
    for h in heads: h.eval()
    model.eval()
    ds = H5Dataset(test_path, 'test')
    dl = DataLoader(ds, shuffle=False, batch_size=bs, num_workers=0)
    all_p, all_id = [], []
    for imgs, _, _, ids in tqdm(dl, desc='Predicting'):
        bp = []
        for ti in range(tta_n):
            batch = []
            for img in imgs:
                proc = img if ti == 0 else _tta((img * 255).clamp(0, 255).to(torch.uint8)).float() / 255.0
                batch.append(transform(proc))
            batch = torch.stack(batch).to(device)
            with torch.no_grad():
                out = model(pixel_values=batch)
                f = out.last_hidden_state[:, 0, :]
                for h in heads:
                    bp.append(h(f).cpu())
        all_p.append(torch.stack(bp).mean(0).squeeze(-1))
        all_id.append(ids)
    del ds; gc.collect()
    ids = torch.cat(all_id).numpy()
    probas = torch.cat(all_p).numpy()
    df = pd.DataFrame({'ID': ids, 'Pred': (probas > 0.5).astype(int)}).set_index('ID')
    df.to_csv(save)
    pd.DataFrame({'ID': ids, 'Proba': probas}).set_index('ID').to_csv(
        save.replace('.csv', '_probas.csv'))
    return df

In [ ]:
# find data and load model
INPUT_DIR = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'train.h5' in files: INPUT_DIR = root; break
assert INPUT_DIR, 'train.h5 not found'
TRAIN = os.path.join(INPUT_DIR, 'train.h5')
VAL   = os.path.join(INPUT_DIR, 'val.h5')
TEST  = os.path.join(INPUT_DIR, 'test.h5')

# try local weights first, fall back to huggingface
WEIGHTS = None
for name in ['phikon_weights', 'phikon-weights']:
    for m in glob.glob(f'/kaggle/input/**/{name}', recursive=True):
        if os.path.isdir(m): WEIGHTS = m; break
    if WEIGHTS: break
if not WEIGHTS:
    for m in glob.glob('/kaggle/working/*phikon*', recursive=True):
        if os.path.isdir(m): WEIGHTS = m; break

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_id = WEIGHTS if WEIGHTS else 'owkin/phikon'
model = ViTModel.from_pretrained(model_id, add_pooling_layer=False).float().to(device)
model.eval()
feat_dim = model.config.hidden_size

n_gpus = torch.cuda.device_count()
if n_gpus > 1:
    model = nn.DataParallel(model)
    BATCH_SIZE *= n_gpus

feat_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
# precompute features (cached)
CACHE = None
for match in glob.glob('/kaggle/input/**/precomputed_*.pt', recursive=True):
    CACHE = match; break
if not CACHE:
    for match in glob.glob('/kaggle/working/precomputed_*.pt'):
        CACHE = match; break

if CACHE:
    c = torch.load(CACHE, weights_only=True)
    tf, tl = c['tf'], c['tl']
    vf, vl = c['vf'], c['vl']
else:
    train_ds = H5Dataset(TRAIN, 'train')
    val_ds = H5Dataset(VAL, 'train')
    key = hashlib.md5(f"{model_id}_noaug_{train_ds.needs_rescale}".encode()).hexdigest()[:8]
    CACHE = f'/kaggle/working/precomputed_{key}.pt'
    train_dl = DataLoader(train_ds, shuffle=False, batch_size=BATCH_SIZE, num_workers=0)
    val_dl = DataLoader(val_ds, shuffle=False, batch_size=BATCH_SIZE, num_workers=0)
    tf, tl = precompute(train_dl, model, feat_transform, device)
    vf, vl = precompute(val_dl, model, feat_transform, device)
    torch.save({'tf': tf, 'tl': tl, 'vf': vf, 'vl': vl}, CACHE)
    del train_ds, val_ds

print(f'Train: {tf.shape}, Val: {vf.shape}')

In [ ]:
# train 5 heads with different seeds (FroFA augmentation)
heads, metrics = [], []
for s in range(N_SEEDS):
    print(f'\n-- Head {s+1}/{N_SEEDS} (seed={s}) --')
    h, m = train_head(tf, tl, vf, vl, feat_dim=feat_dim, device=device, seed=s,
                      lr=LR, wd=WEIGHT_DECAY, bs=BATCH_SIZE,
                      epochs=NUM_EPOCHS, patience=PATIENCE + 5,
                      mixup_alpha=FEAT_MIXUP_ALPHA,
                      feat_dropout=FEAT_DROPOUT,
                      feat_noise=FEAT_NOISE_STD,
                      save=f'/kaggle/working/head_s{s}.pth')
    heads.append(h); metrics.append(m)
print_summary('FroFA ensemble', metrics)

In [ ]:
import matplotlib.pyplot as plt

# per-seed val accuracy
fig, ax = plt.subplots(figsize=(6, 3.5))
accs = [m['val_acc'] * 100 for m in metrics]
ax.bar(range(N_SEEDS), accs, color='#DD8452', edgecolor='black', linewidth=0.5)
ax.axhline(np.mean(accs), color='black', linestyle='--', alpha=0.5, label=f'Mean {np.mean(accs):.2f}%')
ax.set_xlabel('Seed'); ax.set_ylabel('Val Accuracy (%)')
ax.set_xticks(range(N_SEEDS))
ax.set_ylim(min(accs) - 0.3, max(accs) + 0.3)
ax.legend()
plt.tight_layout()
plt.savefig('/kaggle/working/fig_seed_comparison.pdf', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# t-SNE of frozen features coloured by centre and label
train_ds_meta = H5Dataset(TRAIN, 'train')
train_centers = np.array(train_ds_meta.centers)
del train_ds_meta

n_sub = 5000
idx_t = np.random.choice(len(tf), n_sub, replace=False)
all_feats = torch.cat([tf[idx_t], vf]).numpy()
all_labels = torch.cat([tl[idx_t], vl]).numpy()
all_centers = np.concatenate([train_centers[idx_t], np.full(len(vf), 1)])

tsne = TSNE(n_components=2, perplexity=30, random_state=0, n_iter=1000)
emb = tsne.fit_transform(all_feats)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

center_names = {0: 'Centre 0 (train)', 1: 'Centre 1 (val)', 3: 'Centre 3 (train)', 4: 'Centre 4 (train)'}
colors_c = {0: '#4C72B0', 1: '#C44E52', 3: '#55A868', 4: '#8172B2'}
for c in sorted(np.unique(all_centers)):
    mask = all_centers == c
    ax1.scatter(emb[mask, 0], emb[mask, 1], s=1, alpha=0.3, c=colors_c.get(int(c), 'gray'),
                label=center_names.get(int(c), f'Centre {int(c)}'))
ax1.set_title('By centre'); ax1.legend(markerscale=5, fontsize=8)
ax1.set_xticks([]); ax1.set_yticks([])

for l, col, name in [(0, '#4C72B0', 'Non-tumour'), (1, '#C44E52', 'Tumour')]:
    mask = all_labels == l
    ax2.scatter(emb[mask, 0], emb[mask, 1], s=1, alpha=0.3, c=col, label=name)
ax2.set_title('By label'); ax2.legend(markerscale=5, fontsize=8)
ax2.set_xticks([]); ax2.set_yticks([])

plt.tight_layout()
plt.savefig('/kaggle/working/fig_tsne_features.pdf', dpi=150, bbox_inches='tight')
plt.show()
del all_feats, emb; gc.collect()

In [ ]:
# confidence histogram on val set
best_idx = np.argmax([m['val_acc'] for m in metrics])
best_head = heads[best_idx]
best_head.eval()

vloader = DataLoader(PrecomputedDataset(vf, vl), shuffle=False, batch_size=256)
all_proba, all_true = [], []
with torch.no_grad():
    for f, l in vloader:
        p = best_head(f.to(device)).cpu().squeeze(-1)
        all_proba.append(p); all_true.append(l.squeeze(-1))
probas = torch.cat(all_proba).numpy()
trues = torch.cat(all_true).numpy()
correct = ((probas > 0.5).astype(int) == trues)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(probas[correct], bins=50, alpha=0.7, label=f'Correct ({correct.sum()})', color='#55A868', edgecolor='black', linewidth=0.3)
ax.hist(probas[~correct], bins=50, alpha=0.7, label=f'Incorrect ({(~correct).sum()})', color='#C44E52', edgecolor='black', linewidth=0.3)
ax.axvline(0.5, color='black', linestyle='--', alpha=0.5)
ax.set_xlabel('Predicted probability (tumour)'); ax.set_ylabel('Count')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('/kaggle/working/fig_confidence_hist.pdf', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# predict with ensemble
del tf, tl, vf, vl; gc.collect(); torch.cuda.empty_cache()

heads = []
for s in range(N_SEEDS):
    h = nn.Sequential(nn.Linear(feat_dim, 1), nn.Sigmoid()).to(device)
    h.load_state_dict(torch.load(f'/kaggle/working/head_s{s}.pth', weights_only=True))
    heads.append(h)

df = predict_ensemble(model, feat_transform, heads, TEST, device,
                      tta_n=TTA_N, bs=BATCH_SIZE)